## swiGLU 4 - Retraining (global recovery) Analysis

The purpose of this notebook is to test different model retraining configurations and approaches on a smaller retraining (global reocvery) budgets.

Final winning configuration and approach will be then used for production retraining experiments.

setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

In [ ]:
def find_project_root(start):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'mlp_replacement').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root')


PROJECT_ROOT = find_project_root(Path.cwd())
ARTIFACT_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'results'
    / 'workflows'
    / 'model'
    / 'swiglu-4'
    / 'run-001.json'
)

In [ ]:
if not ARTIFACT_PATH.is_file():
    raise FileNotFoundError(f'Artifact does not exist: {ARTIFACT_PATH}')

artifact = json.loads(ARTIFACT_PATH.read_text(encoding='utf-8'))
if artifact.get('workflow') != 'swiglu-4':
    raise ValueError('Expected a swiglu-4 artifact')
if artifact.get('schema_version') != 1:
    raise ValueError('Expected schema version 1')
if artifact.get('status') != 'completed':
    raise ValueError(
        f"Artifact is not completed: {artifact.get('status')}"
    )

configuration = artifact['configuration']
results = artifact['results']
print(ARTIFACT_PATH.relative_to(PROJECT_ROOT))

In [ ]:
run_overview_df = pd.DataFrame([{
    'execution_mode': configuration['execution_mode'],
    'sparsity_key': results['source_state']['sparsity_key'],
    'calibration_pairs': results['source_state']['calibration_pairs'],
    'requested_target_tokens': configuration['recovery']['target_tokens'],
    'effective_batch_tokens': results['data']['effective_batch_tokens'],
    'dense_wikitext_ppl': (
        results['dense_baseline']['wikitext_validation']['perplexity']
    ),
}])
display(run_overview_df)

#### Configuration testing
- learning rate, scheduling
- KL, KL+CE
- weight decay

In [ ]:
configuration_stage = results['configuration_testing']
configuration_trajectories = configuration_stage['trajectories']

candidate_rows = []
for trajectory_id, trajectory in configuration_trajectories.items():
    values = trajectory['configuration']
    candidate_rows.append({
        'id': trajectory_id,
        'label': trajectory['label'],
        'learning_rate': values['learning_rate'],
        'scheduler': values['scheduler'],
        'warmup_fraction': values['warmup_fraction'],
        'temperature': values['temperature'],
        'ce_weight': values['ce_weight'],
        'weight_decay': values['weight_decay'],
        'tokens_seen': trajectory['tokens_seen'],
        'trainable_parameters': (
            trajectory['trainable_scope']['trainable_parameters']
        ),
    })

configuration_candidates_df = (
    pd.DataFrame(candidate_rows)
    .sort_values('id')
    .reset_index(drop=True)
)
display(configuration_candidates_df)

In [ ]:
optimizer_selection_df = pd.DataFrame([
    configuration_stage['optimizer_selection']
])
objective_selection_df = pd.DataFrame([
    configuration_stage['objective_selection']
])
configuration_winner_df = pd.json_normalize([
    configuration_stage['winner']
], sep='.')

display(optimizer_selection_df)
display(objective_selection_df)
display(configuration_winner_df)

In [ ]:
configuration_history_df = pd.DataFrame([
    {'id': trajectory_id, **row}
    for trajectory_id, trajectory in configuration_trajectories.items()
    for row in trajectory['validation_history']
])

figure, axes = plt.subplots(1, 2, figsize=(13, 4))
for trajectory_id, group in configuration_history_df.groupby('id'):
    group = group.sort_values('tokens_seen')
    token_millions = group['tokens_seen'] / 1e6
    axes[0].plot(
        token_millions,
        group['recovery_validation_kl'],
        marker='.',
        label=trajectory_id,
    )
    learning_rate = group['learning_rates'].map(
        lambda values: next(iter(values.values())) if values else None
    )
    axes[1].plot(
        token_millions,
        learning_rate,
        marker='.',
        label=trajectory_id,
    )

axes[0].set(
    xlabel='Actual recovery tokens (M)',
    ylabel='Fixed validation KL (T=1)',
    title='Configuration recovery',
)
axes[1].set(
    xlabel='Actual recovery tokens (M)',
    ylabel='Replacement learning rate',
    title='Learning-rate histories',
)
axes[1].set_yscale('log')
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.show()

In [ ]:
configuration_milestone_rows = []
for trajectory_id, trajectory in configuration_trajectories.items():
    for milestone in trajectory['milestones']:
        current = milestone['current']
        configuration_milestone_rows.append({
            'id': trajectory_id,
            'requested_tokens': milestone['requested_tokens'][0],
            'actual_tokens': milestone['actual_tokens'],
            'validation_kl_t1': current['recovery_validation_kl_t1'],
            'wikitext_ppl': (
                current['wikitext_validation']['perplexity']
            ),
            'allocation_ppl': (
                current['allocation_selection']['perplexity']
            ),
        })

configuration_milestones_df = (
    pd.DataFrame(configuration_milestone_rows)
    .sort_values(['requested_tokens', 'validation_kl_t1'])
    .reset_index(drop=True)
)
display(configuration_milestones_df)

#### RMSNorm

In [ ]:
rms_stage = results['rmsnorm']
rms_trajectories = rms_stage['trajectories']

rms_rows = []
for trajectory_id, trajectory in rms_trajectories.items():
    endpoint = trajectory['endpoint']
    rms_rows.append({
        'id': trajectory_id,
        'label': trajectory['label'],
        'rmsnorm_lr_multiplier': (
            trajectory['configuration']['rmsnorm_learning_rate_multiplier']
        ),
        'actual_tokens': endpoint['actual_tokens'],
        'validation_kl_t1': endpoint['recovery_validation_kl_t1'],
        'wikitext_ppl': (
            endpoint['metrics']['wikitext_validation']['perplexity']
        ),
        'trainable_parameters': (
            trajectory['trainable_scope']['trainable_parameters']
        ),
    })

display(pd.DataFrame([rms_stage['selection']]))
display(pd.DataFrame(rms_rows).sort_values('id').reset_index(drop=True))

In [ ]:
figure, axis = plt.subplots(figsize=(7, 4))
for trajectory_id, trajectory in rms_trajectories.items():
    history_df = (
        pd.DataFrame(trajectory['validation_history'])
        .sort_values('tokens_seen')
    )
    axis.plot(
        history_df['tokens_seen'] / 1e6,
        history_df['recovery_validation_kl'],
        marker='.',
        label=trajectory_id,
    )
axis.set(
    xlabel='Actual recovery tokens (M)',
    ylabel='Fixed validation KL (T=1)',
    title='RMSNorm scope comparison',
)
axis.grid(alpha=0.25)
axis.legend()
plt.show()

In [ ]:
rms_parameter_group_rows = []
for trajectory_id, trajectory in rms_trajectories.items():
    for group in trajectory['trainable_scope']['parameter_groups']:
        rms_parameter_group_rows.append({
            'trajectory': trajectory_id,
            'name': group['name'],
            'learning_rate': group['learning_rate'],
            'weight_decay': group['weight_decay'],
            'parameter_tensors': group['parameter_tensors'],
            'parameters': group['parameters'],
        })

rms_parameter_groups_df = pd.DataFrame(rms_parameter_group_rows)
display(rms_parameter_groups_df)

#### LoRA

mlp subset

In [ ]:
lora_stage = results['lora']
l1 = lora_stage['trajectories']['L1']

l1_summary_df = pd.DataFrame([{
    'id': 'L1',
    'scope': l1['configuration']['variant'],
    'rank': l1['configuration']['lora_rank'],
    'alpha': l1['configuration']['lora_alpha'],
    'adapter_lr': l1['configuration']['lora_learning_rate'],
    'target_modules': len(l1['trainable_scope']['lora_paths']),
    'trainable_parameters': (
        l1['trainable_scope']['trainable_parameters']
    ),
    'validation_kl_t1': l1['endpoint']['recovery_validation_kl_t1'],
    'wikitext_ppl': (
        l1['endpoint']['metrics']['wikitext_validation']['perplexity']
    ),
}])
display(l1_summary_df)

In [ ]:
l1_history_df = (
    pd.DataFrame(l1['validation_history'])
    .sort_values('tokens_seen')
)

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(
    l1_history_df['tokens_seen'] / 1e6,
    l1_history_df['recovery_validation_kl'],
    marker='.',
)
axes[0].set(
    xlabel='Actual recovery tokens (M)',
    ylabel='Fixed validation KL (T=1)',
    title='L1 validation',
)
axes[1].plot(
    l1_history_df['tokens_seen'] / 1e6,
    l1_history_df['mean_train_kl_since_resume'],
    label='KL',
)
axes[1].plot(
    l1_history_df['tokens_seen'] / 1e6,
    l1_history_df['mean_train_loss_since_resume'],
    label='total',
)
axes[1].set(
    xlabel='Actual recovery tokens (M)',
    ylabel='Training loss',
    title='L1 training history',
)
for axis in axes:
    axis.grid(alpha=0.25)
axes[1].legend()
plt.tight_layout()
plt.show()

entire model

In [ ]:
l2 = lora_stage['trajectories']['L2']

l2_summary_df = pd.DataFrame([{
    'id': 'L2',
    'scope': l2['configuration']['variant'],
    'rank': l2['configuration']['lora_rank'],
    'alpha': l2['configuration']['lora_alpha'],
    'adapter_lr': l2['configuration']['lora_learning_rate'],
    'target_modules': len(l2['trainable_scope']['lora_paths']),
    'trainable_parameters': (
        l2['trainable_scope']['trainable_parameters']
    ),
    'validation_kl_t1': l2['endpoint']['recovery_validation_kl_t1'],
    'wikitext_ppl': (
        l2['endpoint']['metrics']['wikitext_validation']['perplexity']
    ),
    'embeddings_lm_head_tied': (
        l2['trainable_scope']['embedding_and_lm_head'][
            'weight_storage_tied'
        ]
    ),
}])
display(l2_summary_df)

In [ ]:
excluded_modules_df = pd.DataFrame([
    {
        'excluded_module': module,
        'reason': (
            'tied vocabulary table is outside the transformer-recovery scope'
        ),
    }
    for module in l2['trainable_scope']['excluded_modules']
])
lora_comparison_df = pd.DataFrame([lora_stage['comparison']])

display(excluded_modules_df)
display(lora_comparison_df)

In [ ]:
figure, axis = plt.subplots(figsize=(7, 4))
for trajectory_id in ('L1', 'L2'):
    trajectory = lora_stage['trajectories'][trajectory_id]
    history_df = (
        pd.DataFrame(trajectory['validation_history'])
        .sort_values('tokens_seen')
    )
    axis.plot(
        history_df['tokens_seen'] / 1e6,
        history_df['recovery_validation_kl'],
        marker='.',
        label=trajectory_id,
    )
axis.set(
    xlabel='Actual recovery tokens (M)',
    ylabel='Fixed validation KL (T=1)',
    title='LoRA scope comparison',
)
axis.grid(alpha=0.25)
axis.legend()
plt.show()

#### Resutls

winner comparison

In [ ]:
comparison_df = (
    pd.DataFrame(results['final_comparison'])
    .sort_values('recovery_validation_kl_t1')
    .reset_index(drop=True)
)
display(comparison_df)

In [ ]:
target_millions = configuration['recovery']['target_tokens'] / 1e6
dense_perplexity = (
    results['dense_baseline']['wikitext_validation']['perplexity']
)

figure, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(
    comparison_df['role'],
    comparison_df['recovery_validation_kl_t1'],
)
axes[0].set(
    ylabel='Fixed validation KL (T=1)',
    title=f'{target_millions:g}M recovery comparison',
)
axes[1].bar(
    comparison_df['role'],
    comparison_df['wikitext_validation_perplexity'],
)
axes[1].axhline(
    dense_perplexity,
    color='black',
    linestyle='--',
    label='dense',
)
axes[1].set(
    ylabel='WikiText-2 validation perplexity',
    title='Quality after recovery',
)
axes[1].legend()
for axis in axes:
    axis.tick_params(axis='x', rotation=55)
    axis.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
selected_trajectories = {
    'C0': (configuration_stage['trajectories'], 'C0'),
    'configuration winner': (
        configuration_stage['trajectories'],
        configuration_stage['winner']['id'],
    ),
    'RMSNorm winner': (
        rms_stage['trajectories'],
        rms_stage['winner']['id'],
    ),
    'L1': (lora_stage['trajectories'], 'L1'),
    'L2': (lora_stage['trajectories'], 'L2'),
}

figure, axis = plt.subplots(figsize=(8, 5))
for label, (trajectories, trajectory_id) in selected_trajectories.items():
    history_df = (
        pd.DataFrame(trajectories[trajectory_id]['validation_history'])
        .sort_values('tokens_seen')
    )
    axis.plot(
        history_df['tokens_seen'] / 1e6,
        history_df['recovery_validation_kl'],
        marker='.',
        label=label,
    )
axis.set(
    xlabel='Actual recovery tokens (M)',
    ylabel='Fixed validation KL (T=1)',
    title='Selected recovery histories',
)
axis.grid(alpha=0.25)
axis.legend()
plt.show()

In [ ]:
runtime_df = pd.DataFrame(results['runtime'])
runtime_df['minutes'] = runtime_df['seconds'] / 60
runtime_df['hours'] = runtime_df['seconds'] / 3600
display(runtime_df)

In [ ]:
provenance_df = pd.DataFrame([{
    'artifact_status': artifact['status'],
    'source_artifact': (
        artifact['provenance']['source_paths']['swiglu_3_artifact']
    ),
    'source_sha256': (
        artifact['provenance']['source_sha256']['swiglu_3_artifact']
    ),
    'packed_token_fingerprint': (
        results['data']['packed_token_cache']['fingerprint']
    ),
    'persistent_weight_assets': (
        artifact['storage']['persistent_weight_assets']
    ),
    'temporary_scratch_removed': (
        artifact['storage']['temporary_scratch_removed']
    ),
}])
display(provenance_df)

In [ ]:
operator_state_files_df = (
    pd.DataFrame(results['data']['operator_state_files'])
    .T
    .rename_axis('layer')
    .reset_index()
)
display(operator_state_files_df)